In [ ]:
import os
import re
import warnings
import random
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, average_precision_score,
    confusion_matrix, brier_score_loss, roc_curve
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import shap
import joblib

# CONFIG
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.preprocessing._encoders")
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

BASE_DIR = Path("/Users/amanda/Desktop/UCBT")

INPUT_PATH = BASE_DIR / "ucbt_dataset.csv" 

OUT_DIR = BASE_DIR / "models_output" / "metrics"
OUT_DIR.mkdir(exist_ok=True, parents=True)

# Dose filters and ALL-only for Survival
#H1
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = True

#H2
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = False

#H3
#USE_DOSE_FILTERS = False
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = True

#H4
#USE_DOSE_FILTERS = False
#CD34_MIN, TNC_MIN = 2.5, 3.5
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = False

#H5
USE_DOSE_FILTERS = True
CD34_MIN, TNC_MIN = 3.0, 4.0
CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
USE_ALL_ONLY_SURVIVAL = True

#H6
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 3.0, 4.0
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 6.5, 8.6 
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 6.8, 9.5
#USE_ALL_ONLY_SURVIVAL = False

#H7
#USE_DOSE_FILTERS = True
#CD34_MIN, TNC_MIN = 2.0, 3.0
#CD34_MAX_PLATELET, TNC_MAX_PLATELET = 7.5, 9.6
#CD34_MAX_SURVIVAL, TNC_MAX_SURVIVAL = 7.8, 10.5
#USE_ALL_ONLY_SURVIVAL = True

# Bagging
N_BAGS = 5

# Cross-validation
N_FOLDS = 10
MIN_SAMPLES_CV = 10

# Minimum group size for plots
MIN_GROUP_N = 20

# SHAP top k features
SHAP_TOP_K = 6

# Holdout set proportion
HOLDOUT_SIZE = 0.2

# LABELS / NAMING
RUN_DATE = pd.Timestamp.now().strftime("%Y%m%d")
DATASET_TAG = "synthetic" if "synthetic" in INPUT_PATH.name.lower() else "real"

def make_safe_filename(s: str) -> str:
    s = re.sub(r"[^\w\-\.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:200] if len(s) > 200 else s

def out_csv(path_stem: str) -> Path:
    """Builds a clean CSV filename in OUT_DIR with dataset tag + date."""
    return OUT_DIR / f"{path_stem}_{DATASET_TAG}_{RUN_DATE}.csv"

# ENV CHECK
print(f"Python executable: {sys.executable}")
try:
    print(f"SHAP version: {shap.__version__}")
except AttributeError:
    print("SHAP not properly installed. Please reinstall: pip install shap==0.46.0")

# UTILITIES
def prevalence(y: np.ndarray) -> float:
    y = np.asarray(y).astype(int)
    return float(np.mean(y))

def best_threshold_for_accuracy(y_true, proba, lo=0.05, hi=0.95, steps=901):
    grid = np.linspace(lo, hi, steps)
    acc = [accuracy_score(y_true, (proba >= t).astype(int)) for t in grid]
    i = int(np.argmax(acc))
    return float(grid[i]), float(acc[i])

def build_preprocessor(cat_cols, num_cols, categories):
    cat_tf = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False))
    ])
    num_tf = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", MinMaxScaler())
    ])
    pre = ColumnTransformer(
        transformers=[
            ("num", num_tf, num_cols),
            ("cat", cat_tf, cat_cols)
        ],
        remainder="drop"
    )
    return pre

def evaluate(y_true, proba, prefix=""):
    thr, acc = best_threshold_for_accuracy(y_true, proba)
    y_pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        f'{prefix}acc': acc,
        f'{prefix}auc': roc_auc_score(y_true, proba),
        f'{prefix}pr_auc': average_precision_score(y_true, proba),
        f'{prefix}brier': brier_score_loss(y_true, proba),
        f'{prefix}thr': thr,
        f'{prefix}prec': (tp / (tp + fp)) if (tp + fp) else 0.0,
        f'{prefix}rec': (tp / (tp + fn)) if (tp + fn) else 0.0,
        f'{prefix}f1': (2 * tp / (2 * tp + fp + fn)) if (2*tp + fp + fn) else 0.0,
        f'{prefix}tn': int(tn), f'{prefix}fp': int(fp),
        f'{prefix}fn': int(fn), f'{prefix}tp': int(tp)
    }

def shap_feature_selection(model, X_tr, colnames, top_k=SHAP_TOP_K):
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_tr)
        shap_sum = np.abs(shap_values).mean(axis=0)
        if len(shap_sum) != len(colnames):
            print(f"Warning: SHAP sum length ({len(shap_sum)}) != len(colnames) ({len(colnames)}); using all features.")
            return colnames, list(range(len(colnames)))
        feature_importance = pd.DataFrame({'feature': colnames, 'importance': shap_sum})
        top_features = feature_importance.nlargest(top_k, 'importance')['feature'].tolist()
        top_indices = [colnames.index(f) for f in top_features if f in colnames]
        print(f"SHAP top features: {top_features}")
        return top_features, top_indices
    except Exception as e:
        print(f"SHAP failed: {e}; using all features.")
        return colnames, list(range(len(colnames)))

def subgroup_block(df, y_true, proba, thr, col, task_name):
    out = []
    if col not in df.columns:
        return pd.DataFrame()
    cats = df[col].astype(str).fillna('NA').value_counts().index.tolist()
    for k in cats:
        idx = (df[col].astype(str).fillna('NA') == k)
        if idx.sum() < MIN_GROUP_N:
            continue
        yt, pt = y_true[idx], proba[idx]
        auc_s = roc_auc_score(yt, pt) if len(np.unique(yt)) > 1 else np.nan
        acc_s = accuracy_score(yt, (pt >= thr).astype(int))
        ap_s = average_precision_score(yt, pt) if len(np.unique(yt)) > 1 else np.nan
        out.append({'group': col, 'level': k, 'n': int(idx.sum()), 'auc': auc_s, 'acc': acc_s, 'ap': ap_s})
    return pd.DataFrame(out)

def plot_roc(model, X, y, title, path):
    prob = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, prob)
    auc = roc_auc_score(y, prob)
    plt.figure(figsize=(5.5, 4.5))
    plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(title); plt.legend()
    plt.tight_layout(); plt.savefig(path, dpi=220, bbox_inches="tight"); plt.close()

def plot_acc_prev_for_group(df_plot, y_true, proba, group_col, thr, title_prefix, out_dir):
    if group_col not in df_plot.columns:
        return
    data = pd.DataFrame({"group": df_plot[group_col].astype(str),
                         "y": y_true.astype(int),
                         "pred": (proba >= thr).astype(int)})
    stats = (data.groupby("group")
                 .agg(n=("y", "size"),
                      prev=("y", "mean"),
                      acc=("pred", lambda s: (s.values == data.loc[s.index, "y"].values).mean()))
                 .reset_index())
    stats = stats.sort_values("prev", ascending=False)
    stats = stats[stats["n"] >= MIN_GROUP_N]
    if stats.empty: return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    plt.suptitle(f"{title_prefix} — {group_col}")
    axes[0].bar(stats["group"], stats["acc"]); axes[0].set_title("Accuracy by Group")
    axes[0].set_xticklabels(stats["group"], rotation=45, ha="right"); axes[0].set_ylim(0, 1)
    for i, (g, acc, n) in enumerate(zip(stats["group"], stats["acc"], stats["n"])):
        axes[0].text(i, acc + 0.02, f"n={n}", ha="center", va="bottom", fontsize=9)
    axes[1].bar(stats["group"], stats["prev"]); axes[1].set_title("Prevalence by Group")
    axes[1].set_xticklabels(stats["group"], rotation=45, ha="right"); axes[1].set_ylim(0, 1)
    for i, (g, prev, n) in enumerate(zip(stats["group"], stats["prev"], stats["n"])):
        axes[1].text(i, prev + 0.02, f"n={n}", ha="center", va="bottom", fontsize=9)
    for ax in axes: ax.grid(axis="y", alpha=0.2)
    fname = make_safe_filename(f"{title_prefix}_acc_prev_{group_col}.png")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]); plt.savefig(out_dir / fname, dpi=220, bbox_inches="tight"); plt.close()
    print(f"[saved] {out_dir / fname}")

# SAFE FIT
def safe_fit_xgb(X_tr, y_tr, params=None):
    params = dict(params or {})
    pos = int(np.sum(y_tr))
    spw = params.get('scale_pos_weight', (len(y_tr) - pos) / max(pos, 1) if pos > 0 else 1.0)
    p = {
        'objective': "binary:logistic",
        'eval_metric': "auc",
        'random_state': params.pop("random_state", RANDOM_STATE),
        'n_jobs': -1,
        'tree_method': "hist",
        'scale_pos_weight': spw
    }
    p.update(params)
    model = XGBClassifier(**p)
    model.fit(X_tr, y_tr, verbose=False)
    return model

# CROSS-VALIDATION (leak-free)
def cv_with_threshold(X_df, y, task_name, outcome_col, cat_cols, num_cols, categories, best_params):
    print(f"[{task_name}] Running {N_FOLDS}-fold CV (leak-free)...")
    if len(X_df) < MIN_SAMPLES_CV or len(np.unique(y)) < 2:
        print(f"[{task_name}] CV skipped: insufficient samples or single class.")
        return {"auc_mean": np.nan, "auc_std": np.nan, "acc_mean": np.nan, "acc_std": np.nan,
                "prec_mean": np.nan, "prec_std": np.nan, "rec_mean": np.nan, "rec_std": np.nan,
                "best_spw": best_params.get('scale_pos_weight', 1.0)}

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    spw_grid = [0.5, 0.75, 1.0, 1.25, 1.5]
    spw_scores, cv_metrics = [], []

    for fold, (tr, te) in enumerate(cv.split(X_df, y), 1):
        X_tr, X_te = X_df.iloc[tr], X_df.iloc[te]
        y_tr, y_te = y[tr].astype(int), y[te].astype(int)

        pre = build_preprocessor(cat_cols, num_cols, categories)
        X_tr_mat = pre.fit_transform(X_tr[cat_cols + num_cols])
        X_te_mat = pre.transform(X_te[cat_cols + num_cols])
        colnames = list(pre.get_feature_names_out())

        # SMOTE train only
        sm = SMOTE(random_state=RANDOM_STATE)
        try:
            X_tr_mat, y_tr = sm.fit_resample(X_tr_mat, y_tr)
        except ValueError:
            pass

        # SHAP select on train only
        xgb_temp = safe_fit_xgb(X_tr_mat, y_tr, params=best_params)
        top_features, top_indices = shap_feature_selection(xgb_temp, X_tr_mat, colnames)
        X_tr_mat = X_tr_mat[:, top_indices]
        X_te_mat = X_te_mat[:, top_indices]

        # tune spw
        best_spw, best_auc = None, -np.inf
        for spw in spw_grid:
            temp_params = {**best_params, 'scale_pos_weight': spw}
            model = safe_fit_xgb(X_tr_mat, y_tr, params=temp_params)
            probs = model.predict_proba(X_te_mat)[:, 1]
            auc = roc_auc_score(y_te, probs)
            spw_scores.append({'fold': fold, 'spw': spw, 'auc': auc})
            if auc > best_auc:
                best_auc, best_spw = auc, spw

        model = safe_fit_xgb(X_tr_mat, y_tr, params={**best_params, 'scale_pos_weight': best_spw})
        proba_te = model.predict_proba(X_te_mat)[:, 1]
        metrics = evaluate(y_te, proba_te, prefix="cv_")
        cv_metrics.append(metrics)
        print(f"[{task_name}] Fold {fold}: AUC={metrics['cv_auc']:.3f}, ACC={metrics['cv_acc']:.3f}, best_spw={best_spw}")

    cv_df = pd.DataFrame(cv_metrics)
    cv_summary = {
        "auc_mean": cv_df["cv_auc"].mean(),
        "auc_std":  cv_df["cv_auc"].std(),
        "acc_mean": cv_df["cv_acc"].mean(),
        "acc_std":  cv_df["cv_acc"].std(),
        "prec_mean": cv_df["cv_prec"].mean(),
        "prec_std":  cv_df["cv_prec"].std(),
        "rec_mean":  cv_df["cv_rec"].mean(),
        "rec_std":   cv_df["cv_rec"].std(),
    }
    spw_df = pd.DataFrame(spw_scores)
    if not spw_df.empty:
        cv_summary["best_spw"] = spw_df.groupby("spw")["auc"].mean().idxmax()
    else:
        cv_summary["best_spw"] = best_params.get("scale_pos_weight", 1.0)

    print(f"[{task_name}] CV Summary: AUC={cv_summary['auc_mean']:.3f}±{cv_summary['auc_std']:.3f}, "
          f"ACC={cv_summary['acc_mean']:.3f}±{cv_summary['acc_std']:.3f}")
    return cv_summary

# MASKS
def dose_filter_mask(d):
    ok = pd.Series(True, index=d.index)
    if "CD34_num" in d.columns:
        s = pd.to_numeric(d["CD34_num"], errors="coerce")
        ok &= (s >= CD34_MIN) & (s <= CD34_MAX_PLATELET) & s.notna()
    if "TNC_num" in d.columns:
        s = pd.to_numeric(d["TNC_num"], errors="coerce")
        ok &= (s >= TNC_MIN) & (s <= TNC_MAX_PLATELET) & s.notna()
    if "Conditioning_Regimen" in d.columns:
        ok &= (d["Conditioning_Regimen"].isin(['MA', 'RIC'])) & d["Conditioning_Regimen"].notna()
    return ok

def all_only_mask(d):
    ok = (d["Disease_Type"] == "ALL") & d["Disease_Type"].notna() & (d["Disease_Type"] != "Unknown")
    if "CD34_num" in d.columns:
        s = pd.to_numeric(d["CD34_num"], errors="coerce")
        ok &= (s >= CD34_MIN) & (s <= CD34_MAX_SURVIVAL) & s.notna()
    if "TNC_num" in d.columns:
        s = pd.to_numeric(d["TNC_num"], errors="coerce")
        ok &= (s >= TNC_MIN) & (s <= TNC_MAX_SURVIVAL) & s.notna()
    if "Conditioning_Regimen" in d.columns:
        ok &= (d["Conditioning_Regimen"].isin(['MA', 'RIC'])) & d["Conditioning_Regimen"].notna()
    if "HLA_Match_Level" in d.columns:
        ok &= (d["HLA_Match_Level"].isin(['4/6', '5/6', '6/6'])) & d["HLA_Match_Level"].notna()
    return ok

def default_train_mask_platelet(d):
    return d["Platelet_Engraftment"].notna() & dose_filter_mask(d) if USE_DOSE_FILTERS else d["Platelet_Engraftment"].notna()

def eval_mask_platelet(d):
    return d["Platelet_Engraftment"].notna() & dose_filter_mask(d) if USE_DOSE_FILTERS else d["Platelet_Engraftment"].notna()

def default_train_mask_survival(d):
    return d["1_Year_Survival"].notna() & all_only_mask(d) if USE_ALL_ONLY_SURVIVAL else d["1_Year_Survival"].notna()

def eval_mask_survival(d):
    return d["1_Year_Survival"].notna() & all_only_mask(d) if USE_ALL_ONLY_SURVIVAL else d["1_Year_Survival"].notna()

# FEATURE SETS
CAT_COLS = ["Disease_Type", "Conditioning_Regimen", "HLA_Match_Level", "Ethnicity", "Race"]
NUM_COLS = ["Recipient_Age", "CD34_num", "TNC_num", "HLAxCD34", "Age_x_CD34", "Age_x_TNC", "Regimen_MA", "Remission_Status"]

# LOAD + PREP
print(f"Loading: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
print("Loaded:", df.shape)

# Derive helper cols if missing
def hla_to_numeric(hla):
    if pd.isna(hla) or hla == 'Unknown':
        return np.nan
    try:
        num, den = map(int, str(hla).split('/'))
        return num / den
    except:
        return np.nan

if 'ROW_ID' not in df.columns:
    df = df.reset_index(drop=True)
    df['ROW_ID'] = np.arange(len(df))

if 'HLAxCD34' not in df.columns and 'HLA_Match_Level' in df.columns and 'CD34_num' in df.columns:
    df['HLA_numeric'] = df['HLA_Match_Level'].apply(hla_to_numeric)
    df['HLAxCD34'] = df['HLA_numeric'] * pd.to_numeric(df['CD34_num'], errors='coerce')
if 'Age_x_CD34' not in df.columns and 'Recipient_Age' in df.columns and 'CD34_num' in df.columns:
    df['Age_x_CD34'] = pd.to_numeric(df['Recipient_Age'], errors='coerce') * pd.to_numeric(df['CD34_num'], errors='coerce')
if 'Age_x_TNC' not in df.columns and 'Recipient_Age' in df.columns and 'TNC_num' in df.columns:
    df['Age_x_TNC'] = pd.to_numeric(df['Recipient_Age'], errors='coerce') * pd.to_numeric(df['TNC_num'], errors='coerce')
if 'Regimen_MA' not in df.columns and 'Conditioning_Regimen' in df.columns:
    df['Regimen_MA'] = (df['Conditioning_Regimen'] == 'MA').astype(int)


# Map categories
category_mappings = {
    "HLA_Match_Level": {
        '4/6,4/6': '4/6', '4/6,5/6': '4/6', '4/6,6/6': '4/6',
        '5/6,5/6': '5/6', '5/6,6/6': '5/6', '6/6,6/6': '6/6',
        '<=4/6': '4/6', '>=4/6': '4/6', '>=5/6': '5/6', '<=5/8': '5/6',
        '4/6 or 5/6': '4/6', '6-8/8': '6/6', 'nan': 'Unknown', '': 'Unknown',
        'Missing/Unknown': 'Unknown', 'Missing': 'Unknown'
    },
    "Disease_Type": {'': 'Unknown', 'nan': 'Unknown'},
    "Conditioning_Regimen": {
        '': 'Unknown', 'nan': 'Unknown', 'Myeloablative': 'MA',
        'Non-myeloablative': 'RIC', 'Reduced intensity': 'RIC', 'Reduced Intensity': 'RIC'
    },
    "Ethnicity": {
        '': 'Unknown', 'nan': 'Unknown', 'Other/Unknown': 'Unknown',
        'Hispanic': 'Hispanic/Latinx', 'Not Hispanic or Latino': 'Non-Hispanic',
        'Hispanic or Latino': 'Hispanic/Latinx', 'Japanese': 'Asian',
        'Other': 'Unknown', 'Unknown/Missing': 'Unknown'
    },
    "Race": {
        '': 'Unknown', 'nan': 'Unknown', 'Missing/Unknown': 'Unknown',
        'Caucasian(white)': 'Caucasian', 'African-American(Black)': 'African American',
        'Non-White': 'Other', 'Non-Caucasian': 'Other', 'Native American': 'Other',
        'Black or African American': 'African American', 'White': 'Caucasian',
        'Black': 'African American', 'Unknown/Missing': 'Unknown',
        'More than one race': 'Other', 'American Indian or Alaskan Native': 'Other',
        'Others': 'Other'
    }
}
for col, mapping in category_mappings.items():
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().replace(mapping)

# Predefined categories
categories = []
for col in ["Disease_Type", "Conditioning_Regimen", "HLA_Match_Level", "Ethnicity", "Race"]:
    if col in df.columns:
        categories.append(sorted(df[col].unique().tolist()))

# TASKS
def default_train_mask_platelet(d):
    return d["Platelet_Engraftment"].notna() & dose_filter_mask(d) if USE_DOSE_FILTERS else d["Platelet_Engraftment"].notna()

def eval_mask_platelet(d):
    return d["Platelet_Engraftment"].notna() & dose_filter_mask(d) if USE_DOSE_FILTERS else d["Platelet_Engraftment"].notna()

def default_train_mask_survival(d):
    return d["1_Year_Survival"].notna() & all_only_mask(d) if USE_ALL_ONLY_SURVIVAL else d["1_Year_Survival"].notna()

def eval_mask_survival(d):
    return d["1_Year_Survival"].notna() & all_only_mask(d) if USE_ALL_ONLY_SURVIVAL else d["1_Year_Survival"].notna()

CAT_COLS = ["Disease_Type", "Conditioning_Regimen", "HLA_Match_Level", "Ethnicity", "Race"]
NUM_COLS = ["Recipient_Age", "CD34_num", "TNC_num", "HLAxCD34", "Age_x_CD34", "Age_x_TNC", "Regimen_MA", "Remission_Status"]

tasks = [
    ("Platelet_Engraftment_FullDose", "Platelet_Engraftment",
     default_train_mask_platelet, eval_mask_platelet, CAT_COLS, NUM_COLS, [], categories),
    ("1_Year_Survival_FullALL", "1_Year_Survival",
     default_train_mask_survival, eval_mask_survival, CAT_COLS, NUM_COLS, [], categories),
]

# RUNNER
def run_task(task_name, outcome_col, train_mask_fn, eval_mask_fn, cat_cols, num_cols, exclude_cols=None, categories=None):
    print(f"\n[{task_name}] Preparing data...")
    exclude_cols = exclude_cols or []
    full_mask = df[outcome_col].notna()
    X_df_full = df[full_mask].copy()

    if outcome_col not in df.columns:
        raise ValueError(f"Outcome '{outcome_col}' not found in data.")

    filtered_mask = train_mask_fn(df) & eval_mask_fn(df)
    X_df_filtered = df[filtered_mask].copy()

    # Drop rows missing any model feature
    X_df_full = X_df_full[X_df_full[outcome_col].notna() & X_df_full[cat_cols + num_cols].notna().all(axis=1)].reset_index(drop=True)
    y_full = X_df_full[outcome_col].astype(int).values
    X_df_filtered = X_df_filtered[X_df_filtered[outcome_col].notna() & X_df_filtered[cat_cols + num_cols].notna().all(axis=1)].reset_index(drop=True)
    y_filtered = X_df_filtered[outcome_col].astype(int).values

    print(f"[{task_name}] Full dataset size: {len(X_df_full)}")
    print(f"[{task_name}] Filtered size: {len(X_df_filtered)}")

    used_cats = [c for c in cat_cols if c in X_df_full.columns and c not in exclude_cols]
    used_nums = [c for c in num_cols if c in X_df_full.columns and c not in exclude_cols]

    # Split filtered -> train / test / holdout
    X_temp, X_te_filtered, y_temp, y_te_filtered = train_test_split(
        X_df_filtered[used_cats + used_nums], y_filtered, test_size=0.2,
        stratify=y_filtered, random_state=RANDOM_STATE
    )
    X_tr_filtered, X_holdout_filtered, y_tr_filtered, y_holdout_filtered = train_test_split(
        X_temp, y_temp, test_size=HOLDOUT_SIZE/(1-HOLDOUT_SIZE), stratify=y_temp, random_state=RANDOM_STATE
    )

    print(f"[{task_name}] Filtered train size: {len(y_tr_filtered)}, Test size: {len(y_te_filtered)}, Holdout size: {len(y_holdout_filtered)}")
    print(f"[{task_name}] Full train size: {len(y_full)}")

    # CV (leak-free)
    best_params = {
        'n_estimators': 1500 if outcome_col == "Platelet_Engraftment" else 1200,
        'max_depth': 4,
        'learning_rate': 0.03 if outcome_col == "Platelet_Engraftment" else 0.07,
        'subsample': 0.85,
        'colsample_bytree': 0.8,
        'min_child_weight': 2,
        'reg_lambda': 1.0,
        'reg_alpha': 0.0,
        'scale_pos_weight': 1.25 if outcome_col == "Platelet_Engraftment" else 1.5
    }
    cv_summary = cv_with_threshold(X_tr_filtered, y_tr_filtered, task_name, outcome_col, used_cats, used_nums, categories, best_params)
    print(f"[{task_name}] Using fixed XGB params: {best_params}")

    # Final preprocessing (fit on train split only)
    pre = build_preprocessor(used_cats, used_nums, categories)
    X_train_mat = pre.fit_transform(X_tr_filtered)
    X_val_mat   = pre.transform(X_holdout_filtered)
    X_test_mat  = pre.transform(X_te_filtered)
    colnames = list(pre.get_feature_names_out())

    # SMOTE train only
    sm = SMOTE(random_state=RANDOM_STATE)
    try:
        X_train_res, y_train_res = sm.fit_resample(X_train_mat, y_tr_filtered)
        print(f"[{task_name}] SMOTE applied: {X_train_res.shape}")
    except ValueError as e:
        print(f"[{task_name}] SMOTE skipped: {e}")
        X_train_res, y_train_res = X_train_mat, y_tr_filtered

    # SHAP feature selection on train only
    xgb_temp = safe_fit_xgb(X_train_res, y_train_res, params=best_params)
    top_features, top_indices = shap_feature_selection(xgb_temp, X_train_res, colnames)

    X_train_res = X_train_res[:, top_indices]
    X_val_mat   = X_val_mat[:, top_indices]
    X_test_mat  = X_test_mat[:, top_indices]

    # Train final (single) model — same as synthetic
    model = safe_fit_xgb(X_train_res, y_train_res, params=best_params)

    # Threshold from holdout
    proba_val = model.predict_proba(X_val_mat)[:, 1]
    thr, _ = best_threshold_for_accuracy(y_holdout_filtered, proba_val)

    # Eval on test / holdout / full
    proba_test = model.predict_proba(X_test_mat)[:, 1]
    eval_metrics_filtered = evaluate(y_te_filtered, proba_test, prefix="filtered_")

    # Full set: transform with same pre + top_indices
    X_full_mat = pre.transform(X_df_full[used_cats + used_nums])[:, top_indices]
    proba_full = model.predict_proba(X_full_mat)[:, 1]
    eval_metrics_full = evaluate(y_full, proba_full, prefix="full_")

    eval_metrics_holdout = evaluate(y_holdout_filtered, proba_val, prefix="holdout_")
    eval_metrics = {}
    eval_metrics.update(eval_metrics_filtered)
    eval_metrics.update(eval_metrics_holdout)
    eval_metrics.update(eval_metrics_full)
    eval_metrics.update(cv_summary)

    print(f"[{task_name}] Filtered AUC={eval_metrics['filtered_auc']:.4f} | ACC={eval_metrics['filtered_acc']:.4f} | thr={eval_metrics['filtered_thr']:.3f}")
    print(f"[{task_name}] Holdout  AUC={eval_metrics['holdout_auc']:.4f}  | ACC={eval_metrics['holdout_acc']:.4f}")
    print(f"[{task_name}] Full     AUC={eval_metrics['full_auc']:.4f}     | ACC={eval_metrics['full_acc']:.4f}")

    # Subgroup tables
    sub_df_filtered = pd.concat([
        subgroup_block(X_te_filtered, y_te_filtered, proba_test,  eval_metrics['filtered_thr'], 'Ethnicity', task_name),
        subgroup_block(X_te_filtered, y_te_filtered, proba_test,  eval_metrics['filtered_thr'], 'Race', task_name)
    ], ignore_index=True)

    sub_df_holdout = pd.concat([
        subgroup_block(X_holdout_filtered, y_holdout_filtered, proba_val, eval_metrics['holdout_thr'], 'Ethnicity', task_name),
        subgroup_block(X_holdout_filtered, y_holdout_filtered, proba_val, eval_metrics['holdout_thr'], 'Race', task_name)
    ], ignore_index=True)

    sub_df_full = pd.concat([
        subgroup_block(X_df_full, y_full, proba_full, eval_metrics['full_thr'], 'Ethnicity', task_name),
        subgroup_block(X_df_full, y_full, proba_full, eval_metrics['full_thr'], 'Race', task_name)
    ], ignore_index=True)

    # Plots
    plot_roc(model, X_test_mat,   y_te_filtered,      f"ROC: {task_name} (Filtered)", OUT_DIR / f"roc_{task_name}_filtered.png")
    plot_roc(model, X_val_mat,    y_holdout_filtered, f"ROC: {task_name} (Holdout)",  OUT_DIR / f"roc_{task_name}_holdout.png")
    plot_roc(model, X_full_mat,   y_full,             f"ROC: {task_name} (Full)",     OUT_DIR / f"roc_{task_name}_full.png")

    # SHAP plots
    try:
        explainer = shap.TreeExplainer(model)
        shap.summary_plot(explainer.shap_values(X_test_mat), X_test_mat, feature_names=top_features, show=False)
        plt.savefig(OUT_DIR / f"shap_summary_{task_name}_filtered.png", dpi=220, bbox_inches="tight"); plt.close()
    except Exception as e:
        print(f"[{task_name}] SHAP plot (filtered) failed: {e}")
    try:
        shap.summary_plot(explainer.shap_values(X_val_mat), X_val_mat, feature_names=top_features, show=False)
        plt.savefig(OUT_DIR / f"shap_summary_{task_name}_holdout.png", dpi=220, bbox_inches="tight"); plt.close()
    except Exception as e:
        print(f"[{task_name}] SHAP plot (holdout) failed: {e}")
    try:
        shap.summary_plot(explainer.shap_values(X_full_mat), X_full_mat, feature_names=top_features, show=False)
        plt.savefig(OUT_DIR / f"shap_summary_{task_name}_full.png", dpi=220, bbox_inches="tight"); plt.close()
    except Exception as e:
        print(f"[{task_name}] SHAP plot (full) failed: {e}")

    # Persist — clean names
    metrics_path = out_csv(f"metrics_{task_name}")
    pd.DataFrame([eval_metrics]).assign(model="XGBoost", task=task_name).to_csv(metrics_path, index=False)
    print(f"[{task_name}] Saved metrics to: {metrics_path}")

    joblib.dump(model, OUT_DIR / f"ucbt_xgb_model_{task_name}.pkl")
    with open(OUT_DIR / f"{task_name}_threshold.json", "w") as f:
        json.dump({"threshold": float(eval_metrics['filtered_thr'])}, f)

    if not sub_df_filtered.empty:
        p = out_csv(f"subgroup_{task_name}_filtered"); sub_df_filtered.to_csv(p, index=False); print(f"[{task_name}] Saved filtered subgroup to: {p}")
    if not sub_df_holdout.empty:
        p = out_csv(f"subgroup_{task_name}_holdout");  sub_df_holdout.to_csv(p, index=False);  print(f"[{task_name}] Saved holdout subgroup to: {p}")
    if not sub_df_full.empty:
        p = out_csv(f"subgroup_{task_name}_full");     sub_df_full.to_csv(p, index=False);     print(f"[{task_name}] Saved full subgroup to: {p}")

    return {
        'task': task_name,
        'outcome': outcome_col,
        'n_train': len(y_full),
        'n_test_filtered': len(y_te_filtered),
        'n_holdout_filtered': len(y_holdout_filtered),
        'n_test_full': len(y_full),
        'filtered_auc': eval_metrics['filtered_auc'],
        'filtered_acc': eval_metrics['filtered_acc'],
        'filtered_thr': eval_metrics['filtered_thr'],
        'filtered_prev': prevalence(y_te_filtered),
        'filtered_ap': eval_metrics['filtered_pr_auc'],
        'filtered_precision': eval_metrics['filtered_prec'],
        'filtered_recall': eval_metrics['filtered_rec'],
        'filtered_f1': eval_metrics['filtered_f1'],
        'holdout_auc': eval_metrics['holdout_auc'],
        'holdout_acc': eval_metrics['holdout_acc'],
        'holdout_thr': eval_metrics['holdout_thr'],
        'holdout_prev': prevalence(y_holdout_filtered),
        'holdout_ap': eval_metrics['holdout_pr_auc'],
        'holdout_precision': eval_metrics['holdout_prec'],
        'holdout_recall': eval_metrics['holdout_rec'],
        'holdout_f1': eval_metrics['holdout_f1'],
        'full_auc': eval_metrics['full_auc'],
        'full_acc': eval_metrics['full_acc'],
        'full_thr': eval_metrics['full_thr'],
        'full_prev': prevalence(y_full),
        'full_ap': eval_metrics['full_pr_auc'],
        'full_precision': eval_metrics['full_prec'],
        'full_recall': eval_metrics['full_rec'],
        'full_f1': eval_metrics['full_f1'],
        'cv_auc_mean': cv_summary['auc_mean'],
        'cv_acc_mean': cv_summary['acc_mean'],
        'cv_auc_std': cv_summary['auc_std'],
        'cv_acc_std': cv_summary['acc_std']
    }

# RUN
rows = []
for name, outcome, tr_mask, ev_mask, cat_cols, num_cols, excl, cats in tasks:
    res = run_task(name, outcome, tr_mask, ev_mask, cat_cols, num_cols, excl, cats)
    rows.append(res)

summary = pd.DataFrame(rows).sort_values(["filtered_auc", "filtered_acc"], ascending=[False, False])
print("\nSummary:")
print(summary[["task", "outcome", "n_train", "n_test_filtered", "n_holdout_filtered",
               "filtered_auc", "filtered_acc", "holdout_auc", "holdout_acc",
               "filtered_thr", "filtered_prev", "filtered_ap", "full_auc", "full_acc",
               "cv_auc_mean", "cv_acc_mean"]])

summary_path = out_csv("summary")
summary.to_csv(summary_path, index=False)
print(f"[saved] {summary_path}")


Python executable: /opt/anaconda3/bin/python
SHAP version: 0.46.0
Loading: /Users/amanda/Desktop/UCBT/ucbt_dataset.csv
Loaded: (10222, 32)

[Platelet_Engraftment_FullDose] Preparing data...
[Platelet_Engraftment_FullDose] Full dataset size: 3769
[Platelet_Engraftment_FullDose] Filtered size: 911
[Platelet_Engraftment_FullDose] Filtered train size: 546, Test size: 183, Holdout size: 182
[Platelet_Engraftment_FullDose] Full train size: 3769
[Platelet_Engraftment_FullDose] Running 10-fold CV (leak-free)...
SHAP top features: ['num__Remission_Status', 'num__Age_x_TNC', 'num__Recipient_Age', 'num__CD34_num', 'num__TNC_num', 'num__HLAxCD34']
[Platelet_Engraftment_FullDose] Fold 1: AUC=0.618, ACC=0.764, best_spw=1.5
SHAP top features: ['num__Remission_Status', 'num__Age_x_TNC', 'num__HLAxCD34', 'num__CD34_num', 'num__TNC_num', 'num__Recipient_Age']
[Platelet_Engraftment_FullDose] Fold 2: AUC=0.365, ACC=0.727, best_spw=1.5
SHAP top features: ['num__Remission_Status', 'num__CD34_num', 'num__TNC